In [ ]:
# --- Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import certifi
import ssl
import os
from dotenv import load_dotenv

load_dotenv()

In [ ]:
# --- SSL Handling
os.environ['SSL_CERT_FILE'] = certifi.where()

from alpha_vantage.timeseries import TimeSeries

In [ ]:
# --- Strategy parameters ---
TICKER       = "SPY"      # change to whatever symbol you like
WINDOW_SHORT = 10         # N: short moving-average window
WINDOW_LONG  = 20         # M: long moving-average window (must be > N)
API_KEY     = os.getenv("ALPHA_VANTAGE_API_KEY")

In [ ]:
# --- Initialize Alpha Vantage API ---
ts = TimeSeries(key=API_KEY, output_format='pandas')

data, meta_data = ts.get_daily(symbol=TICKER, outputsize='full')

daily_ohlcv = data.rename(columns={
    "1. open":   "Open",
    "2. high":   "High",
    "3. low":    "Low",
    "4. close":  "Close",
    "5. volume": "Volume"
})

daily_ohlcv.index = pd.to_datetime(daily_ohlcv.index)
daily_ohlcv = daily_ohlcv.sort_index()

del(data, ts)

In [ ]:
# --- Compute Moving Averages & Signals

# 1) Compute the short and long moving averages
daily_ohlcv["MA_short"] = daily_ohlcv["Close"].rolling(window=WINDOW_SHORT).mean()
daily_ohlcv["MA_long"]  = daily_ohlcv["Close"].rolling(window=WINDOW_LONG ).mean()

# 2) Create a raw crossover signal (1=fast>slow, 0 otherwise)
daily_ohlcv["RawSignal"] = np.where(
    daily_ohlcv["MA_short"] > daily_ohlcv["MA_long"], 1, 0
)

# 3) Shift it by one bar to avoid look-ahead bias
daily_ohlcv["Signal"] = daily_ohlcv["RawSignal"].shift(1).fillna(0)

daily_ohlcv.tail()

In [ ]:
# --- Compute Returns & Equity Curves

# 1) Daily market returns
daily_ohlcv["Market_Returns"] = daily_ohlcv["Close"].pct_change()

# 2) Strategy returns (only when in-market)
daily_ohlcv["Strategy_Returns"] = (
    daily_ohlcv["Market_Returns"] * daily_ohlcv["Signal"]
)

# 3) Build equity curves
daily_ohlcv["Equity_Curve"] = (1 + daily_ohlcv["Strategy_Returns"].fillna(0)).cumprod()
daily_ohlcv["BH_Curve"]     = (1 + daily_ohlcv["Market_Returns"].fillna(0)).cumprod()

# 4) Plot
daily_ohlcv[["Equity_Curve","BH_Curve"]].plot(
    figsize=(10,6),
    title=f"{TICKER} MA Crossover vs. Buy-Hold"
)
plt.ylabel("Growth of $1")
plt.show()

In [ ]:
# --- Performance Metrics

def sharpe(returns, freq=252):
    return (returns.mean() / returns.std()) * np.sqrt(freq)

def max_drawdown(curve):
    peak = curve.cummax()
    drawdown = (curve - peak) / peak
    return drawdown.min()

print("Sharpe (strategy):", sharpe(daily_ohlcv["Strategy_Returns"].dropna()))
print("Max Drawdown:       ", max_drawdown(daily_ohlcv["Equity_Curve"]))

In [ ]:
# --- Parameter Optimization with Optuna
import optuna
from optuna.samplers import RandomSampler, TPESampler

PATIENCE = 500

def early_stop_callback(study, trial):
    # Number of the best‐so‐far trial
    best_no = study.best_trial.number
    # Current trial index
    curr_no = trial.number
    # If we’ve gone PATIENCE trials without a new best, stop
    if curr_no - best_no >= PATIENCE:
        study.stop()

def backtest_metrics(df, N, M):
    data = df.copy()
    data["MAf"]    = data["Close"].rolling(N).mean()
    data["MAl"]    = data["Close"].rolling(M).mean()
    data["Signal"] = (data["MAf"] > data["MAl"]).astype(int).shift(1).fillna(0)
    rets = data["Close"].pct_change() * data["Signal"]
    
    # Sharpe
    sr = rets.mean() / rets.std() * np.sqrt(252)
    # Equity curve
    eq = (1 + rets.fillna(0)).cumprod()
    # Max drawdown
    dd = (eq / eq.cummax() - 1).min()
    # Trade count
    trades = int(data["Signal"].diff().abs().sum())
    
    return sr, dd, trades, eq

# 2) In objective(), unpack the equity and store it
def objective(trial):
    N = trial.suggest_int("N", 1, 100)
    g = trial.suggest_int("g", 1, 30)
    M = trial.suggest_int("M", N + g, 252)

    sr, max_dd, n_trades, eq = backtest_metrics(daily_ohlcv, N, M)

    # 3) Attach the curve for later plotting
    trial.set_user_attr("equity_curve", eq)

    # 4) Enforce robustness
    if n_trades < 30 or max_dd < -0.20:
        return -10.0

    return sr

# 5) Run the study as before
rand_sampler  = RandomSampler(seed=42)
study_rand    = optuna.create_study(direction="maximize", sampler=rand_sampler)
study_rand.optimize(objective, n_trials=1000, n_jobs=-1)

# Continue with TPE, seeding from the same study
tpe_sampler   = TPESampler(seed=42)
study_rand.sampler = tpe_sampler
study_rand.optimize(objective, n_trials=4000, n_jobs=-1, callbacks=[early_stop_callback])
print("Best params:", study_rand.best_params)
print("Best Sharpe:", study_rand.best_value)


In [ ]:
# 1) Compute the buy‐and‐hold equity curve from your price series
bh = (1 + daily_ohlcv["Close"].pct_change().fillna(0)).cumprod()
bh = bh.rename("Buy-Hold")

# 2) Select the trials you want to plot (e.g. top 50 by Sharpe)
top_trials = sorted(study_rand.trials, key=lambda t: t.value, reverse=True)[:1000]

# 3) Extract each trial’s equity curve into a list of Series
equity_curves = []
for t in top_trials:
    eq = t.user_attrs["equity_curve"].rename(f"Trial_{t.number}")
    equity_curves.append(eq)

# 4) Concatenate all Series horizontally into one DataFrame
df_equities = pd.concat(equity_curves + [bh], axis=1)

# 5) (Optional) Inspect the combined DataFrame
print(df_equities.head())

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12,6))
for col in df_equities.columns:
    color = "gold" if col == "Buy-Hold" else "gray"
    lw    = 3 if col == "Buy-Hold" else 1
    alpha = 1 if col == "Buy-Hold" else 0.3
    plt.plot(df_equities.index, df_equities[col], color=color,
             linewidth=lw, alpha=alpha)

plt.title("Equity Curves: Top Trials vs. Buy-Hold")
plt.ylabel("Growth of $1")
plt.show()